# Problem 240: Top Dice

There are $1111$ ways in which five $6$-sided dice (sides numbered $1$ to $6$) can be rolled so that the top three sum to $15$. Some examples are:


$D_1,D_2,D_3,D_4,D_5 = 4,3,6,3,5$\
$D_1,D_2,D_3,D_4,D_5 = 4,3,3,5,6$\
$D_1,D_2,D_3,D_4,D_5 = 3,3,3,6,6$\
$D_1,D_2,D_3,D_4,D_5 = 6,6,3,3,3$


In how many ways can twenty $12$-sided dice (sides numbered $1$ to $12$) be rolled so that the top ten sum to $70$?

In [56]:
from itertools import product


def count_top_sum_bruteforce(n_dice: int, n_sides: int, n_top: int, s: int) -> int:
    result = 0
    for n in product(range(1, n_sides+1), repeat=n_dice):
        _sum = sum(sorted(n, reverse=True)[:n_top])
        if _sum == s:
            result += 1
    return result

print(f"Brute force found {count_top_sum_bruteforce(5, 6, 3, 15)} ways for 5d6 top 3 summing to 15")

Brute force found 1111 ways for 5d6 top 3 summing to 15


In [89]:
from collections import Counter
from itertools import combinations_with_replacement, groupby
from typing import Dict, Iterator, Tuple
from functools import lru_cache
from math import factorial


def count_top_sum(n_dice: int, n_sides: int, n_top: int, target_sum: int) -> int:
    @lru_cache(maxsize=None)
    def generate_constrained_multisets(n_top: int, target_sum: int, distribution: Tuple[int] = None):
        distribution = distribution or (0,) * n_sides
        if n_top < 0 or target_sum < 0 or target_sum > n_top * n_sides:
            return
        if n_top == 0 and target_sum == 0:
            yield {k+1: v for k, v in enumerate(distribution) if v > 0}
        for face_value in range(1, min(n_sides, target_sum) + 1):
            new_distribution = list(distribution)
            new_distribution[face_value-1] += 1
            yield from generate_constrained_multisets(n_top - 1, target_sum - face_value, tuple(new_distribution))
    
    def generate_distributions(remaining_dice: int, max_face_value: int) -> Iterator[Dict[int, int]]:
        """
        Enumerates all multisets (distributions) of 'remaining_dice' dice
        over faces 1..max_face_value using combinations_with_replacement.
        Each yield is {face: count} with positive counts only.
        """
        if remaining_dice < 0 or max_face_value <= 0:
            return
        if remaining_dice == 0:
            yield {}
            return
        if max_face_value == 1:
            yield {1: remaining_dice}
            return

        faces = range(1, max_face_value + 1)
        for combo in combinations_with_replacement(faces, remaining_dice):
            dist = {face: len(list(group)) for face, group in groupby(combo)}
            yield dist
    
    def multinomial(distribution: dict) -> int:
        result = factorial(sum(distribution.values()))
        for count in distribution.values():
            result //= factorial(count)
        return result

    generate_constrained_multisets.cache_clear()

    remaining = n_dice - n_top
    total = 0
    for distribution in generate_constrained_multisets(n_top = n_top, target_sum = target_sum):
        min_face_value = min(distribution.keys())
        for rest_distribution in generate_distributions(remaining_dice = remaining, max_face_value = min_face_value):
            combined_distribution = Counter(distribution)
            combined_distribution.update(rest_distribution)
            total += multinomial(combined_distribution)
    return total

# print(count_top_sum(n_dice=5, n_sides=6, n_top=3, target_sum=15))
print(count_top_sum(n_dice=20, n_sides=12, n_top=10, target_sum=70))

7448717393364181966
